# 02. 직전단가 feature 검증

이미 생성된 `transactions.csv`를 읽어서 `complex_prev_price_per_m2`, `complex_prev_missing`, `prev_deal_gap_days` 품질을 확인합니다. 이 노트북은 CSV를 재생성하지 않습니다.

In [ ]:
from pathlib import Path
import pandas as pd

project_dir = Path("/Users/gwongwangjae/goorm-ai-language-course/final_project")
transactions_path = project_dir / "data" / "processed" / "transactions.csv"
report_path = project_dir / "outputs" / "prev_price_feature_report.md"
summary_path = project_dir / "outputs" / "prev_price_feature_summary.json"
transactions_path.exists(), report_path.exists(), summary_path.exists()


## 1. 검증 리포트 확인


In [ ]:
print(report_path.read_text()[:6000])


## 2. 요약 JSON 확인


In [ ]:
import json
summary = json.loads(summary_path.read_text())
summary.keys()


In [ ]:
summary["grade"], summary["rows"], summary["missing_rate"], summary["gap_quantiles"], summary["ratio_quantiles"]


## 3. 샘플 preview

전체 CSV가 크므로 필요한 컬럼만 일부 행을 읽습니다.

In [ ]:
cols = [
    "transaction_id", "complex_id", "deal_date", "area_m2", "price_per_m2",
    "complex_prev_price_per_m2", "complex_prev_missing", "prev_deal_gap_days"
]
preview = pd.read_csv(transactions_path, usecols=cols, nrows=20)
display(preview)


## 4. 모델 입력용 파생 feature 예시

아래 feature는 모델링 노트북에서 생성해서 쓰고, 원본 `transactions.csv`는 덮어쓰지 않습니다.

In [ ]:
import numpy as np
sample = pd.read_csv(transactions_path, usecols=cols, nrows=100_000)
sample["log_complex_prev_price_per_m2"] = np.where(
    sample["complex_prev_price_per_m2"].notna(),
    np.log(sample["complex_prev_price_per_m2"]),
    np.nan,
)
sample["prev_deal_gap_months"] = sample["prev_deal_gap_days"] / 30.4375
sample["prev_deal_gap_bucket"] = pd.cut(
    sample["prev_deal_gap_days"],
    bins=[0, 30, 90, 180, 365, float("inf")],
    labels=["0-30", "31-90", "91-180", "181-365", "366+"],
)
display(sample.head())
